# Trabajo Práctico: Búsqueda Voraz y A* sobre un Grafo Dirigido

## Imanol Kremis





## 1. El Grafo Dirigido y la Heurística

El problema formal se define como $P = (S, A, T, s_0, G, c)$:
- **Estados:** $\{S, A, B, C, D, G\}$
- **Estado Inicial ($s_0$):** $S$
- **Objetivo ($G$):** $\{G\}$
- **Transiciones y Costos:**
  - $S 	o A$ (costo 2), $S 	o B$ (costo 2)
  - $A 	o C$ (costo 2), $A 	o D$ (costo 5)
  - $B 	o D$ (costo 2)
  - $C 	o G$ (costo 3)
  - $D 	o G$ (costo 6)

- **Función Heurística $h(n)$ (estimación del costo restante hasta $G$):**
  - $h(S) = 7$
  - $h(A) = 5$
  - $h(B) = 7$
  - $h(C) = 3$
  - $h(D) = 6$
  - $h(G) = 0$

- **Regla de Desempate:** Ante prioridades idénticas, se extrae el nodo que se insertó antes en la frontera (**FIFO / orden de inserción estricto**).


In [ ]:
# Representación formal del grafo y la heurística
import heapq
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Grafo dirigido como lista de adyacencia: nodo -> [(sucesor, costo)]
grafo = {
    'S': [('A', 2), ('B', 2)],
    'A': [('C', 2), ('D', 5)],
    'B': [('D', 2)],
    'C': [('G', 3)],
    'D': [('G', 6)],
    'G': []
}

# Heurística h(n) hacia el objetivo G
heuristica = {
    'S': 7,
    'A': 5,
    'B': 7,
    'C': 3,
    'D': 6,
    'G': 0
}

print("Grafo y heurística definidos correctamente.")


In [ ]:
# Visualización del grafo dirigido didáctico
G = nx.DiGraph()

# Agregar aristas con pesos
for origen, vecinos in grafo.items():
    for destino, costo in vecinos:
        G.add_edge(origen, destino, weight=costo)

# Posiciones fijas para visualización clara
pos = {
    'S': (0, 1),
    'A': (1, 2),
    'B': (1, 0),
    'C': (2, 2),
    'D': (2, 0),
    'G': (3, 1)
}

plt.figure(figsize=(9, 5))
node_labels = {nodo: f"{nodo}\nh={heuristica[nodo]}" for nodo in G.nodes()}

nx.draw_networkx_nodes(G, pos, node_size=2200, node_color='#E3F2FD', edgecolors='#1976D2', linewidths=2)
nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=11, font_weight='bold')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=22, edge_color='#455A64', width=2, connectionstyle='arc3,rad=0.05')

edge_labels = {(u, v): f"c={d['weight']}" for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10, font_color='#D32F2F')

plt.title("Grafo Dirigido con Costos de Arista y Valores Heurísticos h(n)", fontsize=13, fontweight='bold', pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()


## 2. Estructura de Datos y Esquema Común de Búsqueda

Cada nodo de búsqueda almacena una tupla/diccionario con la siguiente información:
- `estado`: Identificador del estado actual.
- `padre`: Referencia al nodo antecesor (permite reconstruir el camino por retroceso desde el objetivo hasta la raíz).
- `accion`: Transición efectuada (e.g. `'S -> A'`).
- `g`: Costo real acumulado desde el estado inicial hasta este nodo.
- `h`: Estimación heurística restante desde este nodo hasta el objetivo $G$.
- `f`: Prioridad numérica empleada por la cola con prioridad.

### Esquema Común y Diferencias de Prioridad
Los tres algoritmos comparten:
1. **Cola de prioridad (`heapq`)**: Mantiene la frontera ordenada por prioridad $f$. Para garantizar el **desempate estable por orden de inserción (FIFO)**, insertamos tuplas `(f, insertion_counter, nodo)`.
2. **Diccionario `mejor_g[estado]`**: Almacena el mejor costo $g$ conocido hasta el momento para cada estado.
3. **Descarte de nodos obsoletos**: Al extraer un nodo de la frontera, si `nodo['g'] > mejor_g[estado]`, significa que ya se encontró un camino más barato a ese mismo estado y este nodo se descarta sin expandir.
4. **Comprobación de objetivo al extraer**: La condición `estado == objetivo` se evalúa al **extraer** el nodo de la frontera, nunca al generarlo (fundamental para preservar la garantía de optimalidad en algoritmos basados en costos).
5. **Relajación de aristas y reaperturas**: Al evaluar un sucesor, si su nuevo $g$ es menor que `mejor_g[sucesor]`, se actualiza `mejor_g` y se inserta el nuevo nodo en la frontera. Si el sucesor ya existía previamente en `mejor_g`, se contabiliza una **reapertura/mejora**.

| Algoritmo | Prioridad ($f$) | Criterio de Selección |
| :--- | :---: | :--- |
| **UCS** (Costo Uniforme) | $f = g$ | El camino más barato ya recorrido |
| **Voraz** (Greedy Best-First) | $f = h$ | El estado que parece más cercano a la meta |
| **A\*** | $f = g + h$ | El costo total estimado de la solución |


In [ ]:
# Función de fábrica para crear nodos
def crear_nodo(estado, padre=None, accion=None, g=0, h=0, f=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": f
    }

# Reconstrucción del camino desde los punteros al padre
def reconstruir_camino(nodo_objetivo):
    camino = []
    actual = nodo_objetivo
    while actual is not None:
        camino.append(actual["estado"])
        actual = actual["padre"]
    return camino[::-1]  # Invertir para obtener el orden desde la raíz al objetivo


## 3. Implementación del Motor de Búsqueda

Implementamos un motor general que aplica con exactitud el pseudocódigo solicitado y permite registrar la **traza detallada paso a paso**, así como todas las métricas requeridas:
- `camino`: Lista de estados recorridos desde el inicio hasta el objetivo.
- `costo`: Costo total acumulado $g$ de la solución.
- `estados_generados`: Cantidad total de nodos creados e insertados en la frontera.
- `estados_expandidos`: Cantidad de nodos extraídos de la frontera cuyos sucesores fueron procesados (antes de extraer el objetivo).
- `frontera_maxima`: Tamaño máximo alcanzado por la cola con prioridad en cualquier instante.
- `reaperturas`: Veces que se mejoró el valor de `mejor_g` para un estado que ya había sido descubierto previamente.
- `traza`: Historial paso a paso con el nodo extraído y el estado de la frontera.


In [ ]:
def ejecutar_busqueda(grafo, h_dict, inicio='S', objetivo='G', estrategia='A*'):
    """
    Ejecuta la búsqueda de acuerdo a la estrategia indicada:
    - 'UCS': prioridad f = g
    - 'Voraz': prioridad f = h
    - 'A*': prioridad f = g + h
    
    Respeta:
    - Desempate FIFO estable por orden de inserción mediante un contador creciente.
    - Prueba del objetivo al extraer.
    - Control de repetidos y descarte de nodos obsoletos vía mejor_g.
    """
    frontera = []
    insertion_counter = 0
    mejor_g = {}
    
    # Calcular prioridad inicial según estrategia
    g_ini = 0
    h_ini = h_dict[inicio]
    if estrategia == 'UCS':
        f_ini = g_ini
        prio_label = 'g'
    elif estrategia == 'Voraz':
        f_ini = h_ini
        prio_label = 'h'
    elif estrategia == 'A*':
        f_ini = g_ini + h_ini
        prio_label = 'g+h'
    else:
        raise ValueError(f"Estrategia desconocida: {estrategia}")
        
    raiz = crear_nodo(inicio, padre=None, accion=None, g=g_ini, h=h_ini, f=f_ini)
    mejor_g[inicio] = 0
    heapq.heappush(frontera, (f_ini, insertion_counter, raiz))
    insertion_counter += 1
    
    # Métricas requeridas
    estados_generados = 1
    estados_expandidos = 0
    frontera_maxima = 1
    reaperturas = 0
    traza = []
    
    paso = 0
    while frontera:
        frontera_maxima = max(frontera_maxima, len(frontera))
        prio, _, nodo_actual = heapq.heappop(frontera)
        estado = nodo_actual["estado"]
        g_actual = nodo_actual["g"]
        
        # Descarte de nodo obsoleto si encontramos un camino estrictamente mejor antes
        if g_actual > mejor_g[estado]:
            traza.append({
                "paso": paso,
                "evento": f"DESCARTE (obsoleto: g={g_actual} > mejor_g={mejor_g[estado]})",
                "extraido": estado,
                "prioridad": prio,
                "g": g_actual,
                "h": nodo_actual["h"],
                "frontera": [(f, n["estado"]) for f, _, n in sorted(frontera)]
            })
            continue
            
        paso += 1
        # Captura de la traza en este paso
        traza.append({
            "paso": paso,
            "evento": "EXPANSIÓN" if estado != objetivo else "OBJETIVO ENCONTRADO",
            "extraido": estado,
            "prioridad": prio,
            "g": g_actual,
            "h": nodo_actual["h"],
            "frontera": [(f, n["estado"]) for f, _, n in sorted(frontera)]
        })
        
        # Comprobación de objetivo al extraer
        if estado == objetivo:
            camino_solucion = reconstruir_camino(nodo_actual)
            return {
                "estrategia": estrategia,
                "prioridad_tipo": prio_label,
                "camino": camino_solucion,
                "costo": nodo_actual["g"],
                "estados_expandidos": estados_expandidos,
                "estados_generados": estados_generados,
                "frontera_maxima": frontera_maxima,
                "reaperturas": reaperturas,
                "traza": traza
            }
            
        estados_expandidos += 1
        
        # Relajar sucesores e insertar mejoras
        for sucesor, costo_arista in grafo.get(estado, []):
            nuevo_g = g_actual + costo_arista
            h_suc = h_dict[sucesor]
            
            # Condición de mejora frente a mejor_g
            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:
                if sucesor in mejor_g:
                    reaperturas += 1
                mejor_g[sucesor] = nuevo_g
                
                if estrategia == 'UCS':
                    f_suc = nuevo_g
                elif estrategia == 'Voraz':
                    f_suc = h_suc
                elif estrategia == 'A*':
                    f_suc = nuevo_g + h_suc
                    
                nodo_suc = crear_nodo(
                    estado=sucesor,
                    padre=nodo_actual,
                    accion=f"{estado}->{sucesor}",
                    g=nuevo_g,
                    h=h_suc,
                    f=f_suc
                )
                heapq.heappush(frontera, (f_suc, insertion_counter, nodo_suc))
                insertion_counter += 1
                estados_generados += 1

    return None  # En caso de fracaso


In [ ]:
# Wrappers explícitos para cada uno de los 3 algoritmos
def costo_uniforme(grafo, h_dict, inicio='S', objetivo='G'):
    return ejecutar_busqueda(grafo, h_dict, inicio, objetivo, estrategia='UCS')

def busqueda_voraz(grafo, h_dict, inicio='S', objetivo='G'):
    return ejecutar_busqueda(grafo, h_dict, inicio, objetivo, estrategia='Voraz')

def a_estrella(grafo, h_dict, inicio='S', objetivo='G'):
    return ejecutar_busqueda(grafo, h_dict, inicio, objetivo, estrategia='A*')


In [ ]:
def imprimir_traza(resultado):
    print(f"\n{'='*75}")
    print(f"   TRAZA COMPLETA: {resultado['estrategia']} (Prioridad: {resultado['prioridad_tipo']})")
    print(f"{'='*75}")
    print(f"{'Paso':<6} | {'Evento':<20} | {'Extraído':<8} | {'f':<4} | {'g':<4} | {'h':<4} | {'Frontera tras extraer'}")
    print(f"{'-'*75}")
    for t in resultado['traza']:
        frontera_str = ", ".join([f"({f}, {st})" for f, st in t['frontera']]) or "[]"
        print(f"{t['paso']:<6} | {t['evento']:<20} | {t['extraido']:<8} | {t['prioridad']:<4} | {t['g']:<4} | {t['h']:<4} | {frontera_str}")
    print(f"{'-'*75}")
    print(f"-> Camino hallado: {' -> '.join(resultado['camino'])}")
    print(f"-> Costo total: {resultado['costo']}")
    print(f"-> Estados expandidos antes de extraer G: {resultado['estados_expandidos']}")
    print(f"-> Estados generados: {resultado['estados_generados']}")
    print(f"-> Frontera máxima: {resultado['frontera_maxima']}")
    print(f"-> Reaperturas de estados: {resultado['reaperturas']}")


## 4. Ejecución y Traza de los Algoritmos

### 4.1 Costo Uniforme (UCS)
En UCS la prioridad es exclusivamente el costo acumulado $f(n) = g(n)$.


In [ ]:
res_ucs = costo_uniforme(grafo, heuristica)
imprimir_traza(res_ucs)


### 4.2 Búsqueda Voraz por el Mejor Primero (Greedy Best-First)
En Búsqueda Voraz la prioridad es exclusivamente la estimación heurística al objetivo $f(n) = h(n)$.


In [ ]:
res_voraz = busqueda_voraz(grafo, heuristica)
imprimir_traza(res_voraz)


### 4.3 Búsqueda A*
En A* la prioridad combina el costo recorrido y la heurística restante: $f(n) = g(n) + h(n)$.


In [ ]:
res_astar = a_estrella(grafo, heuristica)
imprimir_traza(res_astar)


## 5. Verificación y Tabla Comparativa

Completamos la tabla comparativa solicitada en la consigna con los resultados obtenidos de la ejecución reproducible.


In [ ]:
# Construcción de la tabla comparativa requerida por la consigna
tabla_comparativa = pd.DataFrame({
    'Resultado': [
        'Camino',
        'Costo',
        'Prioridad',
        'Expandidos antes de extraer G'
    ],
    'UCS': [
        ' -> '.join(res_ucs['camino']),
        res_ucs['costo'],
        res_ucs['prioridad_tipo'],
        res_ucs['estados_expandidos']
    ],
    'Voraz': [
        ' -> '.join(res_voraz['camino']),
        res_voraz['costo'],
        res_voraz['prioridad_tipo'],
        res_voraz['estados_expandidos']
    ],
    'A*': [
        ' -> '.join(res_astar['camino']),
        res_astar['costo'],
        res_astar['prioridad_tipo'],
        res_astar['estados_expandidos']
    ]
})

display(tabla_comparativa.set_index('Resultado'))


In [ ]:
# Tabla complementaria con todas las métricas de rendimiento del espacio de estados
metricas_adicionales = pd.DataFrame({
    'Métrica': [
        'Camino devuelto',
        'Costo de solución',
        'Estados expandidos (antes de extraer G)',
        'Estados generados (insertados en frontera)',
        'Frontera máxima (pico de memoria)',
        'Reaperturas (mejoras a estados conocidos)'
    ],
    'UCS': [
        ' -> '.join(res_ucs['camino']),
        res_ucs['costo'],
        res_ucs['estados_expandidos'],
        res_ucs['estados_generados'],
        res_ucs['frontera_maxima'],
        res_ucs['reaperturas']
    ],
    'Voraz': [
        ' -> '.join(res_voraz['camino']),
        res_voraz['costo'],
        res_voraz['estados_expandidos'],
        res_voraz['estados_generados'],
        res_voraz['frontera_maxima'],
        res_voraz['reaperturas']
    ],
    'A*': [
        ' -> '.join(res_astar['camino']),
        res_astar['costo'],
        res_astar['estados_expandidos'],
        res_astar['estados_generados'],
        res_astar['frontera_maxima'],
        res_astar['reaperturas']
    ]
})

display(metricas_adicionales.set_index('Métrica'))


## 6. Preguntas de Análisis

---

### Pregunta 1: ¿Por qué voraz y A* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?

**Respuesta:**
Voraz y A* coinciden tanto en el camino final devuelto ($S 	o A 	o C 	o G$) como en la cantidad exacta de estados expandidos ($3$ estados: $S$, $A$ y $C$) debido a la conjunción de dos condiciones fundamentales del grafo y la heurística:

1. **Heurística perfectamente informativa y consistente a lo largo del camino óptimo:**
   - La distancia real mínima restante ($h^*(n)$) desde cada nodo del camino óptimo hacia $G$ es:
     - $h^*(S) = 2 + 2 + 3 = 7$
     - $h^*(A) = 2 + 3 = 5$
     - $h^*(C) = 3$
     - $h^*(G) = 0$
   - Los valores de la heurística dada son **exactamente idénticos al costo real**: $h(S)=7, h(A)=5, h(C)=3, h(G)=0$. Es decir, $h(n) = h^*(n)$ sobre esa rama.

2. **Dominancia estricta sobre las ramas alternativas en ambas funciones de evaluación:**
   - Para **Voraz** ($f = h$):
     - En el inicio ($S$): Genera $A$ ($h=5$) y $B$ ($h=7$). Voraz prefiere $A$ porque $5 < 7$.
     - Al expandir $A$: Genera $C$ ($h=3$) y $D$ ($h=6$). En frontera quedan $C(h=3), D(h=6), B(h=7)$. Voraz extrae $C$ porque $3 < 6 < 7$.
     - Al expandir $C$: Genera $G$ ($h=0$). Voraz extrae directamente $G$ ($h=0$).
   - Para **A\*** ($f = g + h$):
     - A lo largo del camino óptimo, como $h(n) = h^*(n)$, el valor de $f(n)$ es constante e igual al costo óptimo total $C^* = 7$:
       - $S$: $f(S) = 0 + 7 = 7$
       - $A$: $f(A) = 2 + 5 = 7$
       - $C$: $f(C) = 4 + 3 = 7$
       - $G$: $f(G) = 7 + 0 = 7$
     - Mientras tanto, los caminos alternativos tienen valores de $f$ estrictamente mayores:
       - $B$: $g=2, h=7 \implies f(B) = 9 > 7$
       - $D$ (vía $A$): $g=7, h=6 \implies f(D) = 13 > 7$
   - Como consecuencia, tanto con $f=h$ como con $f=g+h$, los nodos del camino óptimo $S 	o A 	o C 	o G$ tienen siempre un valor de prioridad menor que cualquier otro nodo en la frontera, provocando que ambos algoritmos sigan exactamente la misma secuencia de expansiones sin explorar ramas secundarias.

---

### Pregunta 2: ¿Garantiza voraz devolver el camino de menor costo en general? Justificá con la propiedad de su prioridad (no con este ejemplo).

**Respuesta:**
**NO**, Búsqueda Voraz por el Mejor Primero **no garantiza en general encontrar el camino de menor costo**.

**Justificación formal basada en su prioridad:**
- La función de prioridad de la búsqueda voraz es $f(n) = h(n)$. Esta función es **miope hacia atrás**: evalúa únicamente la estimación hacia adelante (la proximidad aparente al objetivo) e **ignora por completo el costo acumulado $g(n)$ ya incurrido** para llegar a dicho estado.
- Teóricamente, un algoritmo que solo prioriza $h(n)$ puede ser atraído por una rama que ofrece un descenso rápido de $h$ (pasos locales con heurística pequeña) pero a través de aristas extremadamente costosas ($g(n)$ enorme), o hacia caminos que parecen prometedores inicialmente pero que terminan en un callejón sin salida o requieren un salto final de altísimo costo.
- Para garantizar optimalidad, la función de ordenamiento debe tomar en cuenta el costo real acumulado $g(n)$ y contar con una heurística admisible ($h(n) \le h^*(n)$). Al descartar $g(n)$, el algoritmo voraz pierde toda cota superior sobre el costo de la solución devuelta.

---

### Pregunta 3: ¿Qué ocurre si se usa h = 0 en A*? ¿Con qué algoritmo coincide entonces?

**Respuesta:**
Si se define la función heurística idénticamente nula para todos los estados,
- La función de evaluación de A* pasa a ser:
  $$f(n) = g(n) + h(n) = g(n) + 0 = g(n)$$
- La prioridad de cada nodo se reduce pura y exclusivamente a su **costo acumulado $g(n)$**.
- Por lo tanto, A* coincide **exactamente con la Búsqueda de Costo Uniforme (UCS - Uniform Cost Search)** (que equivale al algoritmo de Dijkstra para grafos generales con costos no negativos).
- En este caso, la búsqueda deja de ser "informada": ya no posee direccionalidad guiada hacia el objetivo y se expande en "ondas concéntricas" de costo creciente hasta alcanzar la meta.

---

### Pregunta 4: ¿Hubo reaperturas en UCS? ¿Y en A* y voraz? ¿Por qué?

**Respuesta:**
- **En UCS:** **SÍ hubo 1 reapertura** (en el estado $D$).
  - *Mecanismo:* Al expandir $A$, se descubrió $D$ con un costo acumulado $g = 2 + 5 = 7$, estableciendo `mejor_g['D'] = 7`. Más adelante, al expandir $B$ ($g=2$), se procesó la arista $B 	o D$ ($c=2$), calculando un nuevo costo $g = 2 + 2 = 4$. Dado que $4 < 7$, se produjo una **mejora de `mejor_g['D']`** (reapertura) y se insertó nuevamente $D$ en la frontera con prioridad $4$. (Posteriormente, cuando la entrada vieja de $D$ con $g=7$ fue extraída de la cola, fue correctamente descartada por ser obsoleta: $7 > 4$).
- **En Voraz:** **NO hubo reaperturas ($0$)**.
  - *Mecanismo:* Voraz se dirigió directamente al objetivo siguiendo la disminución más rápida de $h$ ($S 	o A 	o C 	o G$). Durante esa trayectoria nunca descubrió caminos alternativos con menor costo para estados ya conocidos antes de extraer $G$.
- **En A\*:** **NO hubo reaperturas ($0$)**.
  - *Mecanismo:* En este grafo, la heurística $h$ es **consistente (o monótona)** a lo largo del camino óptimo, satisfaciendo la desigualdad triangular:
    $$h(u) \le c(u, v) + h(v)$$
    Cuando un algoritmo A* opera con una heurística consistente sobre un grafo, se demuestra formalmente que la primera vez que un nodo es extraído para expansión, su costo acumulado $g(n)$ ya es estrictamente óptimo. Por ende, ningún camino posterior puede ofrecer un costo menor para ese nodo, impidiendo reaperturas de nodos expandidos. Además, para los nodos no expandidos como $D$, la búsqueda finalizó al extraer $G$ sin llegar a procesar aristas alternativas hacia $D$.

---

### Pregunta 5: ¿"Expandir menos estados" significa "camino más barato"? Relacionalo con lo que muestran UCS y voraz aquí.

**Respuesta:**
**NO**. Expandir menos estados **no implica en absoluto** que el camino hallado sea más barato.

**Relación con la evidencia empírica observada:**
1. **La coincidencia en este ejemplo particular:**
   - En este grafo, Voraz expandió únicamente $3$ estados ($S, A, C$) mientras que UCS expandió $5$ estados ($S, A, B, C, D$). Casualmente, ambos devolvieron una solución con el mismo costo óptimo ($7$).
2. **La causa real de la diferencia en expansiones:**
   - Voraz expandió menos estados no porque sea "más óptimo", sino porque es **codicioso**: toma decisiones precipitadas eligiendo siempre el vecino que parece más cercano según $h$, sin detenerse a verificar si existen otros caminos más baratos.
   - UCS, en cambio, expandió $5$ estados porque tiene una **garantía matemática de optimalidad**: antes de certificar que un camino al objetivo es el mejor posible, UCS debe explorar exhaustivamente todas las trayectorias de menor costo en todas las direcciones para asegurarse de que ninguna otra alternativa oculta pueda ser más económica.
3. **Generalización:**
   - La cantidad de estados expandidos mide el **esfuerzo computacional (tiempo de búsqueda)**, mientras que el costo acumulado $g$ mide la **calidad de la solución**.
   - Un algoritmo voraz puede expandir muy pocos nodos avanzando a ciegas hacia un objetivo por un camino que resulta ser carísimo (o incluso infinito en ciclos), mientras que algoritmos que exploran más estados garantizan encontrar el camino globalmente óptimo.


## 7. Conclusiones

1. **UCS (Costo Uniforme):** Garantiza encontrar siempre el camino de costo mínimo al guiarse por $g(n)$, a expensas de explorar más estados debido a la falta de información direccional hacia la meta.
2. **Voraz (Greedy Best-First):** Muy rápido y eficiente en cantidad de expansiones cuando la heurística es precisa, pero carece de garantías teóricas sobre la optimalidad del costo encontrado en el caso general.
3. **A\*:** Combina lo mejor de ambos mundos: la garantía de optimalidad de UCS junto con la direccionalidad y eficiencia de Voraz al sumar el costo real $g(n)$ y la heurística admisible/consistente $h(n)$.
4. El experimento demostró con total reproducibilidad que la correcta implementación del desempate FIFO y la prueba del objetivo al extraer son pilares esenciales para respetar los estándares formales de los algoritmos de búsqueda.
